In [209]:
# import necessary packages
import numpy as np
import pandas as pd
import plotly.express as px
import json

# import geojson data
with open("neighborhoods.geojson", "r", encoding="utf-8") as f:
    pgh_nbs = json.load(f)

# import facilities data
df = pd.read_csv("facilities.csv")

# import neighborhood data
nb = pd.read_csv("neighborhoods.csv")

In [210]:
# examine facilities data
df.head()

,_id,id,parcel_id,inactive,name,rentable,type,primary_user,address_number,street,...,neighborhood,council_district,ward,tract,public_works_division,pli_division,police_zone,fire_zone,latitude,longitude
0,1,650726265,120-J-300,f,57th Street Park Building,f,Storage,Department of Public Works,NaN,57TH ST,...,Upper Lawrenceville,7,10.0,42003101100,2.0,10.0,2.0,3-5,40.485666,-79.946450
1,2,783044037,2-H-284,f,Albert Turk Graham Park Shelter,f,Shelter,Department of Public Works,39.0,VINE ST,...,Crawford-Roberts,6,3.0,42003030500,3.0,3.0,2.0,2-1,40.440458,-79.984104
2,3,1997158435,23-R-157,f,Allegheny Northside Senior Center and Hazlett ...,t,Senior,CitiParks,5.0,ALLEGHENY SQ E,...,Allegheny Center,1,22.0,42003562700,1.0,22.0,1.0,1-6,40.453099,-80.005343
3,4,204824684,10-F-198,f,Ammon Recreation Center,f,Pool,CitiParks,2217.0,BEDFORD AVE,...,Bedford Dwellings,6,5.0,42003050900,3.0,5.0,2.0,2-5,40.448735,-79.977856
4,5,472140955,013-K-314,f,Arlington Field Lights Building,f,Utility,CitiParks,0.0,STERLING ST,...,South Side Slopes,3,16.0,42003160800,3.0,16.0,3.0,4-22,40.418152,-79.974471


In [211]:
# get list of all neighborhood names, sorted
names = nb['hood'].unique()
names.sort()
#names # hide full list printout

In [212]:
# filter results to only have official neighborhoods (removes 1 row)
df = df.loc[df["neighborhood"].isin(names)]

# filter results to remove rows where "inactive" value isn't "f"
df_active = df.loc[df["inactive"] == "f"]

df_active.shape # check size of initial df

(400, 22)

In [213]:
# view list of facility types
df_active['type'].unique()

array(['Storage', 'Shelter', 'Senior', 'Pool', 'Utility', 'Activity',
       'Restrooms', 'Service', 'Concession', 'Dugout', 'Pool/Rec',
       'Rec Center', 'Office', 'Pool Closed', 'Firehouse', 'Community',
       'Vacant', 'Cabin', 'Medic Station', 'Training', 'Police',
       'Salt Dome', 'Recycling', 'SERVICE', 'STORAGE', 'POLICE',
       'TRAINING', 'OFFICE'], dtype=object)

In [214]:
# change "type" strings to all have title case
df_active["type"] = df_active["type"].str.title()
df_active["type"].unique()

/tmp/ipykernel_891/2968704548.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



array(['Storage', 'Shelter', 'Senior', 'Pool', 'Utility', 'Activity',
       'Restrooms', 'Service', 'Concession', 'Dugout', 'Pool/Rec',
       'Rec Center', 'Office', 'Pool Closed', 'Firehouse', 'Community',
       'Vacant', 'Cabin', 'Medic Station', 'Training', 'Police',
       'Salt Dome', 'Recycling'], dtype=object)

In [215]:
# we only want facilities that the public can use
# list of facility types to not count
ignore_list = ["Storage","Service","Office","Training",
               "Salt Dome", "Police", "Firehouse", "Recycling",
               "Medic Station", "Pool Closed", "Utility",
               "Dugout", "Vacant"]

In [216]:
# filter to only include facilities useable by the public
df_useful = df_active.loc[~df_active["type"].isin(ignore_list)]
df_useful.shape # check size of remaining df

(187, 22)

In [217]:
# get count of how many facilities of each type are in each neighborhood
# group by neighborhood and type
df_group = df_useful.groupby(["neighborhood","type"])["id"].count()
df_group = df_group.reset_index()
df_group.rename(columns={"id":"count"}, inplace=True)
df_group

,neighborhood,type,count
0,Allegheny Center,Pool,2
1,Allegheny Center,Restrooms,1
2,Allegheny Center,Senior,1
3,Allentown,Activity,1
4,Banksville,Concession,2
...,...,...,...
125,Upper Hill,Rec Center,1
126,West End,Senior,1
127,Westwood,Pool,2
128,Windgap,Concession,1


In [218]:
# first pass, total number of useable facilites per neighborhood
df_total = df_group.groupby('neighborhood')['count'].sum()
df_total = df_total.reset_index()

In [219]:
# properties.hood == name of neighborhood
fig = px.choropleth_map(df_total,geojson=pgh_nbs,
                    color="count",
                    featureidkey="properties.hood",
                    locations="neighborhood", zoom=10,
                    center={"lat":40.4387, "lon":-79.9972},
                    map_style="streets", opacity=0.5,
                    hover_name="neighborhood",
                    hover_data = {'count':True, 'neighborhood':False},
                    labels={'count':'Total'},
                    title="Total Facilities per Neighborhood")
#
fig.update_layout(margin={'r':0,'l':0},
                  title={"xanchor":"center", "x":0.5},
                  coloraxis_colorbar={"orientation":"h",
                                      "xanchor":"center","x":0.5,
                                      "yanchor":"bottom","y":-0.2})
fig.show()

In [220]:
nb.columns

Index(['_id', 'objectid', 'fid_blockg', 'statefp10', 'countyfp10', 'tractce10',
       'blkgrpce10', 'geoid10', 'namelsad10', 'mtfcc10', 'funcstat10',
       'aland10', 'awater10', 'intptlat10', 'intptlon10', 'shape_leng',
       'fid_neighb', 'pghdb_sde_neighborhood_2010_area', 'perimeter',
       'neighbor', 'neighbor_i', 'hood', 'hood_no', 'acres', 'sqmiles',
       'dpwdiv', 'unique_id', 'sectors', 'shape_le_1', 'shape_ar_1',
       'page_number', 'plannerassign', 'globalid', 'created_user',
       'created_date', 'last_edited_user', 'last_edited_date', 'temp',
       'shape_area', 'shape_length'],
      dtype='object')

In [226]:
nb_data = nb[["hood","sqmiles","intptlat10","intptlon10"]]
nb_data.rename(columns={"hood":"neighborhood","intptlat10":"lat","intptlon10":"lon"}, inplace=True)
nb_data

/tmp/ipykernel_891/2341969893.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,neighborhood,sqmiles,lat,lon
0,Point Breeze North,0.301921,+40.4523867,-079.9073195
1,Squirrel Hill North,1.223409,+40.4427301,-079.9435821
2,Garfield,0.457385,+40.4672423,-079.9433700
3,Bedford Dwellings,0.175674,+40.4512373,-079.9745804
4,Knoxville,0.299625,+40.4109357,-079.9931744
...,...,...,...,...
85,Regent Square,0.196219,+40.4328896,-079.8976172
86,Terrace Village,0.324644,+40.4416579,-079.9764621
87,Elliott,0.605890,+40.4400199,-080.0409711
88,South Side Flats,0.924317,+40.4302576,-079.9922422


In [227]:
# create base map of neighborhoods
fig = px.choropleth_map(nb_data,geojson=pgh_nbs,
                    featureidkey="properties.hood",
                    locations="neighborhood", zoom=10,
                    center={"lat":40.4387, "lon":-79.9972},
                    map_style="streets", opacity=0.3,
                    hover_name="neighborhood",
                    hover_data={"neighborhood":False},
                    title="Total Facilities per Neighborhood")

# adjust margins and title position
fig.update_layout(margin={'r':0,'l':0,'b':0},
                  title={"xanchor":"center", "x":0.5})

fig.show()

In [229]:
# merge facilities and neighborhoods data
df_data = pd.merge(df_total, nb_data, on="neighborhood", how="outer")
df_data.rename(columns={"count":"facilities"}, inplace=True)
# assign value of 0 facilities to neighborhoods without any
df_data.loc[df_data['facilities'].isna(),'facilities']=0
# calculate facilities per acre, rounded
df_data['fac_val'] = round(df_data['facilities'] / df_data['sqmiles'], 1)
# preview results
df_data.head()

,neighborhood,facilities,sqmiles,lat,lon,fac_val
0,Allegheny Center,4.0,0.208937,+40.4515765,-080.0053403,19.1
1,Allegheny West,0.0,0.144794,+40.4507843,-080.0144032,0.0
2,Allentown,1.0,0.296998,+40.4191918,-079.9927314,3.4
3,Arlington,0.0,0.479975,+40.4138332,-079.9632700,0.0
4,Arlington Heights,0.0,0.127234,+40.4168800,-079.9615214,0.0


In [230]:
# create scatter plot showing number of facilities
fig2 = px.scatter_map(df_data,
                     size='facilities',
                     lat="lat",lon="lon",
                     zoom=10, center={"lat":40.4387, "lon":-79.9972},
                      map_style="streets")
fig2.show()

In [233]:
# create base map of neighborhoods
fig = px.choropleth_map(nb_data,geojson=pgh_nbs,
                    featureidkey="properties.hood",
                    locations="neighborhood", zoom=10,
                    center={"lat":40.4387, "lon":-79.9972},
                    map_style="streets", opacity=0.3,
                    hover_name="neighborhood",
                    hover_data={"neighborhood":False},
                    title="Facilities per Square Mile")

# adjust margins and title position
fig.update_layout(margin={'r':0,'l':0,'b':0},
                  title={"xanchor":"center", "x":0.5,
                         "subtitle":{"text":"by Neighborhood"}})

# create scatter plot showing number of facilities
fig2 = px.scatter_map(df_data,
                      size='fac_val',
                      hover_name="neighborhood",
                      hover_data={"lat":False,"lon":False,"fac_val":True},
                      labels={"fac_val":"Facilities Value"},
                      lat="lat",lon="lon",
                      zoom=10, center={"lat":40.4387, "lon":-79.9972},
                      map_style="streets")

# combine to create single plot
fig.add_trace(fig2.data[0])
fig.show()